In [4]:
import sys
sys.path.append("/home/rajesh/CSL7100-Project/code")  # adjust path if needed

from etl import run_etl


In [5]:
from config import print_config
from etl import run_etl

print_config()
run_etl()

🔧 CONFIGURATION
PROJECT_ROOT: /mnt/d/IITJ/Course/sem7/BigData/Project
USE_HDFS: False
BASE_PATH: file:///mnt/d/IITJ/Course/sem7/BigData/Project
PATHS:
  raw: file:///mnt/d/IITJ/Course/sem7/BigData/Project/raw
  parquet: file:///mnt/d/IITJ/Course/sem7/BigData/Project/parquet
  graph: file:///mnt/d/IITJ/Course/sem7/BigData/Project/parquet/graph
  checkpoint: file:///mnt/d/IITJ/Course/sem7/BigData/Project/checkpoints
  spark_temp: file:///mnt/d/IITJ/Course/sem7/BigData/Project/spark-temp
💻 Running in LOCAL mode
📥 Loading data from: file:///mnt/d/IITJ/Course/sem7/BigData/Project/raw
✅ Removed nulls


💾 Saving to: file:///mnt/d/IITJ/Course/sem7/BigData/Project/parquet


✅ ETL Completed


In [6]:
#Data Validation (Quick sanity check)

In [7]:
from pyspark.sql import SparkSession
from config import PATHS

spark = SparkSession.builder.getOrCreate()

df = spark.read.parquet(PATHS["parquet"])

df.show(5)
df.printSchema()
print("Total records:", df.count())

+--------------+----------+-------+--------------+
|    reviewerID|      asin|overall|unixReviewTime|
+--------------+----------+-------+--------------+
| AC6AGACAF0W9M|6300182304|    5.0|    1407110400|
|A3D63MQMPNY1Q8|0793906091|    4.0|    1375228800|
| AUC1FPPBMLGPE|0792838955|    5.0|    1419811200|
|A1PWJICB6QEQ54|0783225857|    5.0|    1505952000|
|A3T35ITD5GMNWU|0783229569|    5.0|    1422835200|
+--------------+----------+-------+--------------+
only showing top 5 rows

root
 |-- reviewerID: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- overall: double (nullable = true)
 |-- unixReviewTime: long (nullable = true)



[Stage 2:>                                                          (0 + 8) / 8]

Total records: 3410019


In [12]:
print("Total records:", df.count())

ConnectionRefusedError: [Errno 111] Connection refused

In [8]:
#Basic Statistics
df.describe().show()

[Stage 5:============================================>              (6 + 2) / 8]

+-------+--------------------+--------------------+------------------+--------------------+
|summary|          reviewerID|                asin|           overall|      unixReviewTime|
+-------+--------------------+--------------------+------------------+--------------------+
|  count|             3410019|             3410019|           3410019|             3410019|
|   mean|                null| 5.509913886106568E9| 4.221320174462371|1.3845512740540156E9|
| stddev|                null|2.0372923441617045E9|1.1664558754885517|1.1585143827730387E8|
|    min|A0009988MRFQ3TROTQPI|          0000143588|               1.0|           881020800|
|    max|       AZZZMSZI9LKE6|          B01HJ1INB0|               5.0|          1538352000|
+-------+--------------------+--------------------+------------------+--------------------+



In [ ]:
from filter_5core import run_5core

run_5core()

In [11]:
from encode_ids import run_encoding

run_encoding()

💻 Running in LOCAL mode


ConnectionRefusedError: [Errno 111] Connection refused

In [13]:
from etl import create_spark_session

spark = create_spark_session()



df = spark.read.parquet(PATHS["parquet"] + "/encoded")
df.show(5)
df.printSchema()

💻 Running in LOCAL mode
💻 Running in LOCAL mode
+-------+-------+-------+--------------+
|user_id|item_id|overall|unixReviewTime|
+-------+-------+-------+--------------+
|  16266|      0|    5.0|    1128556800|
|  18958|      0|    5.0|    1080259200|
|  18933|      0|    5.0|    1255219200|
|  14779|      0|    4.0|    1392249600|
|  18638|      0|    4.0|    1071878400|
+-------+-------+-------+--------------+
only showing top 5 rows

root
 |-- user_id: long (nullable = true)
 |-- item_id: long (nullable = true)
 |-- overall: double (nullable = true)
 |-- unixReviewTime: long (nullable = true)



In [35]:
from graph_builder import build_graph

df = spark.read.parquet(PATHS["parquet"] + "/encoded")

vertices, edges = build_graph(spark, df)

vertices.show(5)
edges.show(5)

Max user_id: 20224


+-----+----+
|   id|type|
+-----+----+
| 7747|user|
| 2509|user|
|17703|user|
|14846|user|
| 4894|user|
+-----+----+
only showing top 5 rows

+-----+-----+------+
|  src|  dst|weight|
+-----+-----+------+
|16266|20225|   5.0|
|18958|20225|   5.0|
|18933|20225|   5.0|
|14779|20225|   4.0|
|18638|20225|   4.0|
+-----+-----+------+
only showing top 5 rows



In [40]:
from graph_builder import run_graph_builder
run_graph_builder()

💻 Running in LOCAL mode
📥 Loading: file:///mnt/d/IITJ/Course/sem7/BigData/Project/parquet/encoded
Max user_id: 20224
💾 Saving vertices...
💾 Saving edges...
✅ Graph Construction Completed


In [41]:
from etl import create_spark_session

spark = create_spark_session()

vertices = spark.read.parquet(PATHS["parquet"] + "/graph/vertices")
edges = spark.read.parquet(PATHS["parquet"] + "/graph/edges")

💻 Running in LOCAL mode


In [42]:
# 1. User exists
vertices.filter(col("id") == source_user).show()

# 2. User is type=user
vertices.filter((col("id") == source_user) & (col("type") == "user")).show()

# 3. User has edges
edges.filter(col("src") == source_user).show()

+-----+----+
|   id|type|
+-----+----+
|16266|user|
+-----+----+

+-----+----+
|   id|type|
+-----+----+
|16266|user|
+-----+----+

+-----+-----+------+
|  src|  dst|weight|
+-----+-----+------+
|16266|20225|   5.0|
|16266|20227|   1.0|
|16266|20467|   4.0|
|16266|20718|   5.0|
|16266|22149|   3.0|
|16266|22498|   3.0|
|16266|23124|   5.0|
|16266|23946|   4.0|
|16266|23975|   5.0|
|16266|25534|   5.0|
|16266|26177|   4.0|
|16266|26442|   3.0|
|16266|26647|   4.0|
|16266|27756|   4.0|
|16266|27868|   4.0|
|16266|27936|   4.0|
|16266|27976|   3.0|
|16266|28386|   5.0|
|16266|28705|   5.0|
|16266|28833|   4.0|
+-----+-----+------+
only showing top 20 rows



In [43]:
source_user = edges.select("src").distinct().limit(1).collect()[0][0]

In [44]:
ranks = personalized_pagerank(spark, vertices, edges, source_user)
ranks.orderBy("rank", ascending=False).show(10)

Iteration 1
Iteration 2
Iteration 3
Iteration 4
Iteration 5
Iteration 6
Iteration 7
Iteration 8
Iteration 9
Iteration 10


[Stage 11:>                                                         (0 + 1) / 1]

+---+----+
| id|rank|
+---+----+
+---+----+



In [23]:
from ppr import personalized_pagerank
from pyspark.sql import SparkSession
from config import PATHS

spark = SparkSession.builder.getOrCreate()

# Load graph
vertices = spark.read.parquet(PATHS["parquet"] + "/graph/vertices")
edges = spark.read.parquet(PATHS["parquet"] + "/graph/edges")

# Choose a user
source_user = 10

ranks = personalized_pagerank(
    spark,
    vertices,
    edges,
    source_id=source_user,
    alpha=0.15,
    max_iter=10
)

ranks.orderBy("rank", ascending=False).show(10)

Iteration 1
Iteration 2
Iteration 3
Iteration 4
Iteration 5
Iteration 6
Iteration 7
Iteration 8
Iteration 9
Iteration 10


[Stage 5:>                                                          (0 + 1) / 1]

+---+----+
| id|rank|
+---+----+
+---+----+

